In [ ]:
import pandas as pd

# ======================================= #
### Load Datasets ###
# ======================================= #

df_causes = pd.read_csv('/data/Principal Causes of Death.csv') #from MoH principal causes of death & percentage
df_totals = pd.read_csv('/data/Total Number of Deaths.csv') #from MoH the actual number of deaths
df_cancer = pd.read_csv('/data/Agestandardisedincidencerateofthetopthreecancersbygenderand5yearperiod.csv') #from HpB

# ======================================= #
### Clean data  ###
# ======================================= #

## Causes & Totals DF + Merge

# Some entry has different case force all to a title case
df_causes["disease_condition"] = df_causes["disease_condition"].str.strip().str.title()

# Some data still have plural and singular. convert the singular to plural

name_mapping = {
    'Ischaemic Heart Diseases': 'Ischaemic Heart Disease',
    'Cerebrovascular Diseases (Including Stroke)': 'Cerebrovascular Disease (Including Stroke)',
    'Cerebrovascular Diseases': 'Cerebrovascular Disease (Including Stroke)',
    'Other Heart Diseases': 'Other Heart Diseases',
    'Hypertensive Diseases (Including Hypertensive Heart Disease)': 'Hypertensive Diseases'
}

df_causes["disease_condition"] = df_causes["disease_condition"].replace(name_mapping)
#print(df_causes["disease_condition"].unique())

#The 2 data sets have a common key which is year. combine only those data that have the years on both data. INNER JOIN
df_deaths_merged = pd.merge(df_causes,df_totals,on ="year", how ="inner")

#Compute the actual nuber of deaths based on % of deaths and no_of_deaths(the overall for that year).
#Create new Column to store. make integer
df_deaths_merged["death_volume"] = ( df_deaths_merged["no_of_deaths"] *  df_deaths_merged["percentage_deaths"] / 100).round().astype(int)

#drop rank icd and classficiation - irrelevant
df_deaths_dropped = df_deaths_merged.drop(columns=['rank','icd', 'classification'])

#there will be a problem where initially there are going to have separate rows because they had diff icd or classification.
#need to combine the total of those together
df_deaths_clean = df_deaths_dropped.groupby(['year', 'disease_condition']).agg({
    'percentage_deaths': 'sum',
    'death_volume': 'sum',
    'no_of_deaths': 'first' # Keeps the total baseline constant
}).reset_index()

## Clean Incidence of cancer DF
df_cancer['site'] = df_cancer['site'].str.strip().str.capitalize()
df_cancer['gender'] = df_cancer['gender'].str.strip().str.capitalize()
df_cancer['time_period'] = df_cancer['period_starting'].astype(str) + " - " + df_cancer['period_ending'].astype(str)
df_cancer_clean = df_cancer[['time_period', 'period_starting', 'period_ending', 'site', 'gender', 'asir']]

# ======================================= #
### Export the 2 CSV  ###
# ======================================= #

df_deaths_clean.to_csv('clean_singapore_deaths_trends.csv', index=False)
df_cancer_clean.to_csv('clean_singapore_cancer_incidence.csv', index=False)
